# Embedding Models: Semantic Similarity, Robustness, and Performance

This notebook explores how different **embedding models** capture semantic meaning
and how model choice impacts retrieval quality, paraphrase robustness,
and performance in Retrieval-Augmented Generation (RAG) systems.
##Objective

The objective of this notebook is to understand:

- How different embedding models represent semantic meaning
- How model choice affects similarity-based retrieval
- How embeddings behave under paraphrased queries
- Performance trade-offs between API-based and open-source models
## What This Notebook Contains

This notebook includes:

- Selection of two embedding models for comparison
- Paragraph-level document chunk design
- Embedding generation using OpenAI and Hugging Face models
- Semantic similarity-based retrieval
- Robustness testing using paraphrased queries
- Performance comparison between models
- Practical observations and trade-offs
## Document Chunk Design

Paragraph-level text chunks are used instead of single sentences.

Reason:
- Real-world RAG systems embed paragraphs, not isolated sentences
- Paragraphs provide richer semantic context
- Improves evaluation of semantic understanding
- Reduces noise caused by overly short text units

Each paragraph represents a distinct topic to minimize semantic overlap.


##Importing Required Libraries

This cell imports libraries for:
- Numerical computation
- Embedding generation
- Semantic similarity calculation
- Performance measurement
- OpenAI API access


In [ ]:
import os
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import openai


In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
OPENAI_BASE_URL = userdata.get("OPENAI_BASE_URL")
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)


## Embedding Model Selection

Two embedding models are selected to compare trade-offs:

1. **text-embedding-3-small**
   - API-based proprietary model
   - Optimized for semantic retrieval
   - Commonly used in production RAG systems

2. **all-mpnet-base-v2**
   - Open-source Hugging Face model
   - Runs locally
   - No API cost


In [ ]:
model_openai = "text-embedding-3-small"
model_hf_name = "all-mpnet-base-v2"
model_hf = SentenceTransformer(model_hf_name)


## Text Chunks

This cell defines paragraph-level document chunks.
Each chunk represents a distinct topic for semantic retrieval.


In [ ]:
chunks = [
    """Artificial intelligence is rapidly transforming the healthcare industry...""",
    """Climate change is one of the most critical challenges facing the world today...""",
    """Renewable energy sources such as solar and wind power help reduce carbon emissions..."""
]


## OpenAI Embedding Function

This function generates embeddings using the OpenAI embedding API.
It returns numerical vectors representing semantic meaning.


In [ ]:
def get_openai_embeddings(texts):
    response = client.embeddings.create(
        model=model_openai,
        input=texts
    )
    return np.array([d.embedding for d in response.data])


## Generating Embeddings Using Both Models

Both models embed the same document chunks.

The following details are printed:
- Model name
- Embedding dimension
- Number of chunks embedded


In [ ]:
emb_openai = get_openai_embeddings(chunks)
emb_hf = model_hf.encode(chunks)

print("----- Part A: Model Info -----")
print(f"Model: {model_openai}, Dimension: {emb_openai.shape[1]}, Chunks: {len(chunks)}")
print(f"Model: {model_hf_name}, Dimension: {emb_hf.shape[1]}, Chunks: {len(chunks)}")


## Query Embedding and Semantic Similarity

A user query is embedded and compared with document embeddings
using cosine similarity to measure relevance.


In [ ]:
query = "How does renewable energy help reduce environmental damage?"

query_openai = get_openai_embeddings([query])
query_hf = model_hf.encode([query])

scores_openai = cosine_similarity(query_openai, emb_openai)[0]
scores_hf = cosine_similarity(query_hf, emb_hf)[0]


In [ ]:
print("\n--- OpenAI Model Ranking ---")
for idx in np.argsort(scores_openai)[::-1]:
    print(f"Score: {scores_openai[idx]:.4f} | Chunk: {chunks[idx]}")

print("\n--- HuggingFace Model Ranking ---")
for idx in np.argsort(scores_hf)[::-1]:
    print(f"Score: {scores_hf[idx]:.4f} | Chunk: {chunks[idx]}")


## Paraphrased Query Evaluation

This cell evaluates how embeddings respond to paraphrased queries.


In [ ]:
paraphrased_query = "How do clean energy technologies protect the environment?"

pq_openai = get_openai_embeddings([paraphrased_query])
pq_hf = model_hf.encode([paraphrased_query])

pq_scores_openai = cosine_similarity(pq_openai, emb_openai)[0]
pq_scores_hf = cosine_similarity(pq_hf, emb_hf)[0]


##  Performance Comparison

This section compares:
- Single embedding latency
- Batch embedding latency

for both OpenAI and Hugging Face models.


In [ ]:
start = time.time()
get_openai_embeddings([chunks[0]])
openai_single = time.time() - start

start = time.time()
model_hf.encode([chunks[0]])
hf_single = time.time() - start

start = time.time()
get_openai_embeddings(chunks)
openai_batch = time.time() - start

start = time.time()
model_hf.encode(chunks)
hf_batch = time.time() - start


##Practical Trade-offs Observed

API-Based Models:
- Higher semantic consistency
- Better paraphrase robustness
- API latency and cost involved

Open-Source Models:
- Faster local execution
- No API cost
- Easier deployment and control

Model choice depends on system constraints and scale.
